In [13]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
data = pd.read_csv('../data/processed/matches.csv')
data.head()

,date,home_team,away_team,tournament,city,country,neutral,year,month,day,...,stadium_temperature_max,stadium_temperature_min,stadium_precipitation,stadium_wind_speed,home_stadium_temp_max,home_stadium_temp_min,home_stadium_temp_avg,away_stadium_temp_max,away_stadium_temp_min,away_stadium_temp_avg
0,1994-06-18,Italy,Republic of Ireland,FWC,East Rutherford,United States,True,1994,6,18,...,32.2,21.3,0.4,11.2,6.7,6.8,6.75,12.8,9.3,11.05
1,1994-06-19,Belgium,Morocco,FWC,Orlando,United States,True,1994,6,19,...,30.1,21.9,6.1,11.0,7.2,10.6,8.90,4.4,6.0,5.20
2,1994-06-21,Argentina,Greece,FWC,Foxborough,United States,True,1994,6,21,...,24.9,16.8,1.0,17.9,10.9,8.4,9.65,10.9,4.9,7.90
3,1994-06-21,Germany,Spain,FWC,Chicago,United States,True,1994,6,21,...,26.7,22.3,1.7,21.9,5.0,12.8,8.90,3.7,5.7,4.70
4,1994-06-22,Romania,Switzerland,FWC,Pontiac,United States,True,1994,6,22,...,27.8,15.2,0.0,10.2,3.0,1.5,2.25,5.0,3.6,4.30


In [15]:
data.drop(columns=['tournament', 'city', 'country', 
                   'year', 'date', 'home_capital', 'away_capital', 
                   'home_comb', 'away_comb', 'stadium_comb', 'home_team', 'away_team',
                   "home_lon", "home_lat", "away_lon", "away_lat",
                   "stadium_lon", "stadium_lat"], inplace=True)

In [16]:
data.columns

Index(['neutral', 'month', 'day', 'result', 'home_points', 'away_points',
       'home_stadium_distance_km', 'away_stadium_distance_km', 'home_fix_1',
       'home_fix_2', 'home_shots_1', 'home_shots_2', 'home_shots_against_1',
       'home_shots_against_2', 'home_scored_1', 'home_scored_2',
       'home_conceded_1', 'home_conceded_2', 'home_relative_shots_1',
       'home_relative_shots_2', 'home_relative_goals_1',
       'home_relative_goals_2', 'away_fix_1', 'away_fix_2', 'away_shots_1',
       'away_shots_2', 'away_shots_against_1', 'away_shots_against_2',
       'away_scored_1', 'away_scored_2', 'away_conceded_1', 'away_conceded_2',
       'away_relative_shots_1', 'away_relative_shots_2',
       'away_relative_goals_1', 'away_relative_goals_2', 'home_ranking',
       'away_ranking', 'home_temperature_max', 'home_temperature_min',
       'home_precipitation', 'home_wind_speed', 'away_temperature_max',
       'away_temperature_min', 'away_precipitation', 'away_wind_speed',
       's

In [17]:
data.select_dtypes(include=['object']).columns

Index([], dtype='str')

In [18]:
data["home_stadium_wind_speed_avg"] = (data["home_stadium_wind_speed_max"] + data["home_stadium_wind_speed_min"]) / 2
data["away_stadium_wind_speed_avg"] = (data["away_stadium_wind_speed_max"] + data["away_stadium_wind_speed_min"]) / 2

data["stadium_temperature_avg"] = (data["home_stadium_temperature"] + data["away_stadium_temperature"]) / 2

KeyError: 'home_stadium_wind_speed_max'

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
train, val = train_test_split(data, test_size=0.2, random_state=42)

# Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
X_train = train.drop(columns=['result'])
y_train = train['result']
X_val = val.drop(columns=['result'])
y_val = val['result']
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
scaled_train = pd.concat([
    pd.DataFrame(X_train, columns=train.drop(columns=['result']).columns),
    y_train.reset_index(drop=True)
], axis=1)

scaled_val = pd.concat([
    pd.DataFrame(X_val, columns=val.drop(columns=['result']).columns),
    y_val.reset_index(drop=True)
], axis=1)

# Feature Selection

## Correlation

In [21]:
columns = scaled_train.columns
for i in range(len(columns)):
    for col in columns[i + 1:]:
        if scaled_train[columns[i]].corr(scaled_train[col]) > 0.75:
            print("High correlation between {} and {} with {}".format(columns[i], col, scaled_train[columns[i]].corr(scaled_train[col])))

In [20]:
scaled_train.drop(columns=["home_points", "away_points", "home_shots_2",
                   "home_shots_1", "away_shots_1", "away_shots_2",
                   "home_temperature_max", "home_temperature_min",
                   "away_temperature_max", "away_temperature_min",
                   "stadium_temperature_max", "stadium_temperature_min",
                   "home_stadium_temp_max", "home_stadium_temp_min",
                   "away_stadium_temp_max", "away_stadium_temp_min"], inplace=True)

# Feature Selection

## Variance Threshold

In [22]:
from sklearn.feature_selection import RFECV

## RFE

In [29]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

In [30]:
X = scaled_train.drop(columns=["result"])
y = scaled_train["result"]

In [31]:
estimator_cv = RandomForestClassifier(random_state=42)
rfecv = RFECV(
    estimator=estimator_cv,
    step=1,
    cv=StratifiedKFold(5),      # 5 folds × n_features iterations
    scoring="accuracy"
)
rfecv.fit(X, y)

selected_features_rfecv = X.columns[rfecv.support_].tolist()

print(f"Total CV iterations: {X.shape[1] * 5}")   # n_features × n_folds
print(f"Optimal n_features:  {rfecv.n_features_}")
print(f"\nSelected features ({len(selected_features_rfecv)}):")
for f in selected_features_rfecv:
    print(f"  ✓ {f}")

print(f"\nCV accuracy per n_features (mean):")
cv_scores = pd.Series(rfecv.cv_results_['mean_test_score'],
                      index=range(1, X.shape[1] + 1))
print(cv_scores.to_string())
print(f"\nBest score: {cv_scores.max():.4f} at n={cv_scores.idxmax()} features")

Total CV iterations: 195
Optimal n_features:  27

Selected features (27):
  ✓ day
  ✓ home_stadium_distance_km
  ✓ away_stadium_distance_km
  ✓ home_fix_1
  ✓ home_fix_2
  ✓ home_shots_against_1
  ✓ home_shots_against_2
  ✓ home_relative_shots_1
  ✓ home_relative_shots_2
  ✓ home_relative_goals_1
  ✓ home_relative_goals_2
  ✓ away_fix_1
  ✓ away_fix_2
  ✓ away_shots_against_1
  ✓ away_shots_against_2
  ✓ away_relative_shots_1
  ✓ away_relative_shots_2
  ✓ home_ranking
  ✓ away_ranking
  ✓ home_precipitation
  ✓ home_wind_speed
  ✓ away_precipitation
  ✓ away_wind_speed
  ✓ stadium_precipitation
  ✓ stadium_wind_speed
  ✓ home_stadium_temp_avg
  ✓ away_stadium_temp_avg

CV accuracy per n_features (mean):
1     0.361293
2     0.378204
3     0.386805
4     0.389542
5     0.429228
6     0.436410
7     0.432055
8     0.450365
9     0.453201
10    0.463200
11    0.458915
12    0.468844
13    0.460324
14    0.457497
15    0.454690
16    0.461782
17    0.457547
18    0.461752
19    0.468914
20